# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print("Dataset Name:", metadata['name'])
print("Description:", metadata['description'])
print("Published:", metadata.get('datePublished', 'N/A'))
print("License:", metadata.get('license', 'N/A'))
print("Dataset Identifier:", metadata.get('identifier', 'N/A'))
print("Keywords:", ', '.join(metadata.get('keywords', [])))

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant metadata contains the record sets and their fields, all uniquely referenced by their `@id`. We list the record sets, fields, and columns below.


In [ ]:
# Collect record set and field information from the Croissant metadata
record_sets = metadata.get('recordSet', [])
if not record_sets:
    print("No record sets found in metadata.")
else:
    for record_set in record_sets:
        record_set_id = record_set['@id']
        print(f"RecordSet @id: {record_set_id}")
        print(f"RecordSet Name: {record_set.get('name', 'N/A')}")
        fields = record_set.get('field', [])
        if fields:
            for field in fields:
                field_id = field['@id']
                print(f"  Field @id: {field_id}")
                print(f"  Field Name: {field.get('name', 'N/A')}")
                columns = field.get('column', [])
                for column in columns:
                    column_id = column['@id']
                    print(f"    Column @id: {column_id}")
                    print(f"    Column Name: {column.get('name', 'N/A')}")
        print('-' * 40)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all available record sets
all_record_set_ids = [rs['@id'] for rs in metadata.get('recordSet', [])]
dataframes = {}

for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show available columns for the first record set
if all_record_set_ids:
    main_record_set = all_record_set_ids[0]
    print(f"Columns for record set {main_record_set}:")
    print(dataframes[main_record_set].columns.tolist())
    display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Operations include removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# Example: Analyze the GAD-7 depression score field (assuming column '@id' is 'gad_score')
# You should substitute the below field @id with actual column @ids from the overview step

main_record_set_id = all_record_set_ids[0] if all_record_set_ids else None

# Example field and group identifiers - update as appropriate from metadata
numeric_field_id = None
group_field_id = None

# Try to locate relevant numeric fields for EDA
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Attempt to find GAD-7 score column
    gad_columns = [c for c in df.columns if 'gad' in c.lower() or 'score' in c.lower()]
    if gad_columns:
        numeric_field_id = gad_columns[0]
    # Use 'gender', 'age', or 'village' for grouping
    possible_group_fields = [c for c in df.columns if c.lower() in ['gender', 'village', 'level_of_education', 'age']]
    if possible_group_fields:
        group_field_id = possible_group_fields[0]

if numeric_field_id:
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id, as_index=False)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field suitable for EDA found. Please check column names above.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for GAD-7 (or similar) score
if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Boxplot of score by group field (e.g., gender or village)
if main_record_set_id and numeric_field_id and group_field_id:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=dataframes[main_record_set_id][group_field_id], y=dataframes[main_record_set_id][numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey dataset is AI-ready and contains rich demographic and psychometric data.
- Loading and processing with `mlcroissant` is seamless via Croissant schema URLs using entity `@id` references.
- Exploratory analysis on GAD-7 (anxiety) scores demonstrates variation potentially linked to demographic variables (e.g., gender, village).
- The schema enables any field or column to be referenced and manipulated using its unique `@id` for reproducible workflows.